# Day 2 Instructor Demo: vectors, RAG, and RAG stacks (Sections 1, 2, 4)

**Run by the instructor only** during the talk sections; shared with participants afterwards.
Cell numbers match the slide tags (D1.1 …). Needs the same secrets as the code-along, including `EMBED_MODEL`.

**Rehearsal tip:** run D2.1 the day before and note what your model claims without RAG.

In [ ]:
import importlib.util
if importlib.util.find_spec("paper_agent") is None:      # Colab: install the course package (about 1 minute)
    !pip install -q "paper-agent[rag] @ git+https://github.com/mari-muthu-k/agent-to-production@main#subdirectory=paper-summarizer"

import logging, os, re, time, json
import numpy as np
import openai
from paper_agent.config import load_env
from paper_agent.llm_client import LLMClient, LLMConfig, summarize_calls
from paper_agent.ingest.embeddings import OpenAIEmbedder
from paper_agent.openrouter import make_client, is_openrouter, REASONING_OFF

# Colab Secrets (🔑) or .env: LLM_API_KEY, LLM_BASE_URL, LLM_MODEL, EMBED_MODEL (+ optional LLM_FALLBACK_MODEL)
# OpenRouter: LLM_BASE_URL=https://openrouter.ai/api/v1, e.g. EMBED_MODEL=nvidia/nemotron-3-embed-1b:free
env = load_env(required=("LLM_API_KEY", "LLM_BASE_URL", "LLM_MODEL", "EMBED_MODEL"))
MODEL, EMBED_MODEL = env["LLM_MODEL"], env["EMBED_MODEL"]
PRICE_IN_PER_1M, PRICE_OUT_PER_1M = 0.50, 2.00   # USD per 1M tokens (illustrative: the instructor gives real ones)
PRICE_EMBED_PER_1M = 0.02
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s", force=True)

client = make_client(env["LLM_API_KEY"], env["LLM_BASE_URL"])     # OpenAI SDK; works with OpenRouter
REASONING = REASONING_OFF if is_openrouter() else None   # short grounded answers don't need long thinking
llm = LLMClient(LLMConfig(model=MODEL, fallback_model=env.get("LLM_FALLBACK_MODEL") or None,
                          price_in_per_1m=PRICE_IN_PER_1M, price_out_per_1m=PRICE_OUT_PER_1M,
                          max_tokens_param=env.get("LLM_MAX_TOKENS_PARAM") or "max_tokens",
                          reasoning=REASONING), client=client)
embedder = OpenAIEmbedder(EMBED_MODEL, client=client, price_per_1m=PRICE_EMBED_PER_1M)

from paper_agent.papers import fetch_paper, paper_ids, paper_info, edge_case_pdf
PAPERS = {pid: fetch_paper(pid) for pid in paper_ids()}      # 4 real papers, CC BY 4.0

print("✅ ready | chat:", MODEL, "| embeddings:", EMBED_MODEL)

---
## D1 Recap: text becomes vectors  ·  *slide: "Similar meaning, nearby vectors"*

### D1.1 Cosine similarity between AI phrases

In [ ]:
phrases = ["sliding window attention", "attention over a sliding window", "KV cache memory",
           "GPU memory used by the KV cache", "pizza with extra cheese"]
V = embedder.embed(phrases)
sims = V @ V.T                                  # unit vectors: dot product = cosine similarity
print(" " * 36 + "".join(f"{i:>6}" for i in range(len(phrases))))
for i, p in enumerate(phrases):
    print(f"{i} {p[:33]:33}" + "".join(f"{s:6.2f}" for s in sims[i]))

Point at: the paraphrases score high, pizza scores near zero. Ask the room: *what would score high that shouldn't?* (word overlap without shared meaning).

### D1.2 What a vector actually is

In [ ]:
v = V[0]
print("dimensions:", v.shape[0], "| length:", round(float(np.linalg.norm(v)), 3))
print("first 8 numbers:", np.round(v[:8], 3))
a, b = V[0], V[1]
print(f"cosine by hand: {float(a @ b) / (np.linalg.norm(a) * np.linalg.norm(b)):.3f}  "
      f"vs dot product of unit vectors: {float(a @ b):.3f}")

---
## D2 What RAG is  ·  *slide: "Retrieve, then answer"*

### D2.1 Without RAG: ask about a paper the model never saw

In [ ]:
q = "In the Ragas paper, how often did Ragas agree with human annotators on faithfulness?"
r = client.chat.completions.create(model=MODEL, max_tokens=120, temperature=0,
                                   messages=[{"role": "user", "content": q}])
print("🤖 (no context):", r.choices[0].message.content)

### D2.2 With RAG: retrieve, then answer from the pages, with citations

In [ ]:
from paper_agent.ingest.pipeline import ingest_pdf
from paper_agent.ingest.store import NumpyStore
from paper_agent.rag.retrieve import Retriever, show_hits
from paper_agent.rag.answer import answer_question
store = NumpyStore(EMBED_MODEL)
for pid, path in PAPERS.items():
    ingest_pdf(path, embedder, store, paper_id=pid)
retriever = Retriever(store, embedder, k=5)
show_hits(retriever.search(q, k=3))
answer_question(q, retriever, llm, min_score=0.0).show()

### D2.3 Grounding includes saying "I don't know"

In [ ]:
answer_question("How many GPUs were used to train Mistral 7B?", retriever, llm, min_score=0.0).show()
answer_question("What is the best pizza topping?", retriever, llm).show()

---
## D4 RAG stacks  ·  *slide: "Pick the boring option first"*

| Layer | Simple / local | Production options | Notes |
|---|---|---|---|
| **Vector index** | numpy (exact search) | Chroma, FAISS, pgvector, Qdrant, Weaviate, Pinecone, Azure AI Search | Exact search is fine to ~100k chunks |
| **Embeddings** | the same endpoint as chat | provider APIs, self-hosted sentence-transformers | One model per index |
| **Splitters / loaders** | pdfplumber + LangChain splitters | unstructured, cloud document AI | Parsing quality caps everything after it |
| **Orchestration** | 30 lines of Python | LangChain, LlamaIndex, Haystack | Frameworks save glue code, not thinking |

### D4.1 Same query, two stores, same answer

In [ ]:
from paper_agent.ingest.store import ChromaStore
import tempfile
chroma = ChromaStore(EMBED_MODEL, path=tempfile.mkdtemp())
chroma.add(store.chunks, store.matrix, embed_model=EMBED_MODEL)
query = embedder.embed(["How does PagedAttention reduce KV cache waste?"])[0]
for name, s in [("numpy ", store), ("Chroma", chroma)]:
    t = time.perf_counter()
    hits = s.search(query, k=3)
    print(f"{name}: {(time.perf_counter() - t) * 1000:6.1f} ms  ->", [(h.chunk.id, round(h.score, 3)) for h in hits])

Identical results: a vector DB is an index plus persistence, filtering and scale. Start with the boring option; move when a number (latency, size, ops) tells you to.